In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Wrap_to_mic)=
# Wrap to MIC

*Wrapping atomic coordinates into the Minimum Image Convention (MIC) box.*

The function {func}`molsysmt.pbc.wrap_to_mic` folds coordinates into the minimum image convention centered on a reference selection (such as a solute macromolecule).

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.pbc.wrap_to_mic`.
:::


## Basic usage

Let's wrap a solvated peptide system into the minimum image convention centered on the peptide solute:


In [2]:
import molsysmt as msm
import numpy as np


In [3]:
molsys = msm.convert(msm.systems['Met-enkephalin']['met_enkephalin.h5msm'])
molsys_solvated = msm.build.solvate(molsys, box_shape='truncated octahedral', clearance='14.0 angstroms', engine='PDBFixer')


:::{note}
:class: dropdown

**Demo Systems Catalog**

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


We wrap the solvent coordinates to the minimum image convention around the peptide while preserving covalent bonds:


In [4]:
molsys_mic = msm.pbc.wrap_to_mic(
    molsys_solvated,
    center_of_selection='molecule_type=="peptide"',
    compact='component'
)


We visualize the wrapped solvated system in 3D using MolSysViewer:


In [5]:
molsysviewer_htmlfile = '_static/views/tools_pbc_wrap_to_mic_1.html'


In [6]:
msm.view(molsys_mic)


'<iframe src="../../../../_static/views/tools_pbc_wrap_to_mic_1.html" width="100%" height="480px"\n        style="border:none;"></iframe>'

## Centering and editing the original

`center_of_selection` defines the moving reference population and must belong
to `selection`. Centering is evaluated before wrapping. With `in_place=True`,
the final result is written to the original system and the return value is
`None`. Otherwise you receive an independent result. The source boxes and
unselected atoms/structures remain unchanged. Native MolSys coordinate setters
invalidate interaction coverage only for the edited structures.

This bundled example edits structures 2 and 0; structure 1 remains untouched.
A centered independent output reuses its working copy rather than copying the
complete system a second time.


In [7]:
molsys = msm.convert(msm.systems['pentalanine']['traj_pentalanine.h5msm'],
    structure_indices=[0, 8, 3])
before = msm.pyunitwizard.get_value(msm.get(molsys, coordinates=True), to_unit='nm').copy()
result = msm.pbc.wrap_to_mic(molsys, selection=[0, 1, 2, 3],
    center_of_selection=[0, 1], structure_indices=[2, 0],
    compact=False, in_place=True)
after = msm.pyunitwizard.get_value(msm.get(molsys, coordinates=True), to_unit='nm')
assert result is None
assert np.array_equal(after[1], before[1])
assert not np.array_equal(after[[2, 0], :4], before[[2, 0], :4])
assert np.array_equal(after[:, 4:], before[:, 4:])


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems with {func}`molsysmt.basic.convert`.
- {ref}`Solvate <Tutorial_Solvate>`: Solvate molecular systems with {func}`molsysmt.build.solvate`.
- {ref}`View <Tutorial_View>`: Render 3D molecular views with {func}`molsysmt.basic.view`.
- {ref}`Wrap to PBC <Tutorial_Wrap_to_pbc>`: Wrap coordinates to primary box with {func}`molsysmt.pbc.wrap_to_pbc`.
- {ref}`Unwrap <Tutorial_Unwrap>`: Restore spatial continuity with {func}`molsysmt.pbc.unwrap`.
:::
